In [28]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Google hesabınızla yetkilendirme yapın
auth.authenticate_user()

# Project ID tanımlama
PROJECT_ID = "project-84485539-b01a-418a-932"
client = bigquery.Client(project=PROJECT_ID)
print("BigQuery bağlantısı başarılı!")

BigQuery bağlantısı başarılı!


big query yetki verildi

In [29]:
table_ref = f"{PROJECT_ID}.{DATASET_NAME}.{TABLE_NAME}"
table = client.get_table(table_ref)
print([field.name for field in table.schema])

['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']


sütün isimleri için yazıldı

In [30]:
DATASET_NAME = "Instacart_Raw_Dataset"
TABLE_NAME = "orders"

query = f"""
SELECT
    user_id,
    -- Recency / Sipariş Aralığı: Müşterinin ortalama kaç günde bir sipariş verdiği
    COALESCE(AVG(days_since_prior_order), 0) AS avg_days_between_orders,

    -- Frequency / Sıklık: Müşterinin toplam sipariş sayısı
    MAX(order_number) AS total_orders,

    -- Davranışsal Metrik 1: Siparişlerin haftanın hangi günlerinde yoğunlaştığı (0-6)
    AVG(order_dow) AS avg_order_dow,

    -- Davranışsal Metrik 2: Günün hangi saatlerinde sipariş verildiği (0-23)
    AVG(order_hour_of_day) AS avg_order_hour

FROM `{PROJECT_ID}.{DATASET_NAME}.{TABLE_NAME}`
WHERE user_id IS NOT NULL
GROUP BY user_id
HAVING total_orders > 1
"""


df_rfm = client.query(query).to_dataframe(create_bqstorage_client=False)

print(f"Başarılı! Toplam {len(df_rfm)} müşteri verisi çekildi.")
df_rfm.head()

Başarılı! Toplam 206209 müşteri verisi çekildi.


,user_id,avg_days_between_orders,total_orders,avg_order_dow,avg_order_hour
0,181070,3.934783,93,2.860215,11.505376
1,174520,7.723404,48,3.479167,13.541667
2,10886,4.556962,80,3.150000,13.062500
3,43990,10.085714,36,3.055556,14.361111
4,146339,10.428571,36,2.277778,13.472222


BigQuery'den çekildi. Müşterilerin ortalama sipariş aralıkları, toplam sipariş sayıları ve alışveriş yaptıkları gün/saat ortalamaları net şekilde tabloya yansımışyansıtıldı

In [31]:

features = ['avg_days_between_orders', 'total_orders', 'avg_order_dow', 'avg_order_hour']


df_log = df_rfm[features].copy()
df_log['total_orders'] = np.log1p(df_log['total_orders'])
df_log['avg_days_between_orders'] = np.log1p(df_log['avg_days_between_orders'])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(df_log)

df_scaled = pd.DataFrame(rfm_scaled, columns=features)
print("Veri başarıyla ölçeklendirildi!")
df_scaled.head()

Veri başarıyla ölçeklendirildi!


,avg_days_between_orders,total_orders,avg_order_dow,avg_order_hour
0,-2.302494,2.681685,0.118506,-1.029263
1,-1.111414,1.801361,0.811851,-0.021567
2,-2.054237,2.480553,0.443121,-0.258692
3,-0.610380,1.421783,0.337325,0.383950
4,-0.546699,1.421783,-0.533936,-0.055933


veri ölçeklendırıldı Çıktıdaki eksi ve artı değerler, tüm verilerin ortalaması 0 ve standart sapması 1 olacak şekilde normalize edildi K-Means mesafeleri hesaplarken hiçbir değişkene torpil geçmeyecek.

In [ ]:
wcss = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, kmeans.labels_))


fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 1. Elbow Grafiği
ax[0].plot(K_range, wcss, marker='o', color='b', linewidth=2)
ax[0].set_title('Dirsek Yöntemi (Elbow Method)')
ax[0].set_xlabel('Küme Sayısı (K)')
ax[0].set_ylabel('WCSS (Küme İçi Toplam Hata)')
ax[0].grid(True)

# 2. Silhouette Grafiği
ax[1].plot(K_range, silhouette_scores, marker='s', color='g', linewidth=2)
ax[1].set_title('Silhouette Skorları')
ax[1].set_xlabel('Küme Sayısı (K)')
ax[1].set_ylabel('Silhouette Skoru')
ax[1].grid(True)

plt.tight_layout()
plt.show()

çok uzun surdu ornekleme yapacagım

In [ ]:

np.random.seed(42)
sample_indices = np.random.choice(rfm_scaled.shape[0], size=min(10000, rfm_scaled.shape[0]), replace=False)
rfm_scaled_sample = rfm_scaled[sample_indices]

wcss = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled_sample)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled_sample, kmeans.labels_))

# Grafikleri Çizdirme
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Elbow Grafiği
ax[0].plot(K_range, wcss, marker='o', color='b', linewidth=2)
ax[0].set_title('Dirsek Yöntemi (Elbow Method)')
ax[0].set_xlabel('Küme Sayısı (K)')
ax[0].set_ylabel('WCSS')
ax[0].grid(True)

# Silhouette Grafiği
ax[1].plot(K_range, silhouette_scores, marker='s', color='g', linewidth=2)
ax[1].set_title('Silhouette Skorları (Örneklem)')
ax[1].set_xlabel('Küme Sayısı (K)')
ax[1].set_ylabel('Silhouette Skoru')
ax[1].grid(True)

plt.tight_layout()
plt.show()

Hem matematiksel dirsek kırılımı hem de iş etiketi verimliliği açısından projeniz için $K = 4$ ideal küme sayısı

10.000 kişilik rastgele örneklem

In [ ]:

OPTIMAL_K = 4


kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df_rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)


cluster_summary = df_rfm.groupby('Cluster').agg(
    customer_count=('user_id', 'count'),
    avg_days_between_orders=('avg_days_between_orders', 'mean'),
    avg_total_orders=('total_orders', 'mean'),
    avg_order_dow=('avg_order_dow', 'mean'),
    avg_order_hour=('avg_order_hour', 'mean')
).reset_index()


cluster_summary['percentage'] = (cluster_summary['customer_count'] / len(df_rfm)) * 100

print("--- MÜŞTERİ KÜME ÖZETİ ---")
display(cluster_summary.round(2))

Instacart veri seti üzerinde yürütülen K-Means kümeleme çalışması sonucunda müşteri kitlemiz 4 temel davranışsal kümeye ayrılmıştır.

Her bir kümenin müşteri büyüklüğü, karakteristik özellikleri ve önerilen pazarlama aksiyonları aşağıda açıklanmıştır:

 1. Küme (Cluster 0): Sadık & Yüksek Hacimli MüşterilerMüşteri Büyüklüğü: 57.674 Kişi (%27.97) — En büyük müşteri grubuKarakteristik Özellikler:Sipariş Sıklığı: Yaklaşık 8 günde bir sipariş vererek platformu çok aktif kullanırlar.Sipariş Hacmi: Müşteri başına ortalama 35.81 sipariş ile diğer tüm kümelerden yaklaşık 4 kat daha fazla alışveriş yapmışlardır.Zamanlama: Gün ortasında (13:20 civarı) ve haftanın ortalama günlerinde dengeli bir alışveriş düzenine sahiptirler.Pazarlama Aksiyonu: İşletmenin ana gelir kaynağını oluşturan bu kitleye VIP sadakat programları sunulmalı, yeni ürün lansmanlarında öncelik verilmeli ve elde tutma (retention) stratejileri uygulanmalıdır

  2. Küme (Cluster 1): Akşamüstü AlışverişçileriMüşteri Büyüklüğü: 48.916 Kişi (%23.72)Karakteristik Özellikler:Sipariş Sıklığı & Hacmi: Ortalama 17.81 günde bir sipariş verirler ve ortalama sipariş sayıları 9.20'dir.Zamanlama: Alışveriş yapma saatleri günün geç saatlerine (15:10 - 16:00 civarı) kaymaktadır. Haftanın ortasından sonuna doğru alışveriş yapmayı tercih ederler.Pazarlama Aksiyonu: Saat 14:00 - 15:30 arasında gönderilecek anlık mobil bildirimler (push notifications) ve akşamüstü saatlerine özel fırsatlar ile sipariş dönüşüm oranları artırılabilir.
  
  3. Küme (Cluster 2): Erken Saat / Sabah AlışverişçileriMüşteri Büyüklüğü: 50.896 Kişi (%24.68)Karakteristik Özellikler:Sipariş Sıklığı & Hacmi: Ortalama 18.24 günde bir sipariş verirler, toplam sipariş ortalamaları 9.23'tür.Zamanlama: Günün erken saatlerinde (11:30 civarı) sipariş vermeyi tercih ederler.Pazarlama Aksiyonu: Öğleden önce saat 09:00 - 11:00 arasında "Günün İndirimi", taze gıda veya kahvaltılık ürün odaklı e-posta/SMS kampanyaları ile hedeflenmelidirler.
  
  4. Küme (Cluster 3): Hafta Başı AlışverişçileriMüşteri Büyüklüğü: 48.723 Kişi (%23.63)Karakteristik Özellikler:Sipariş Sıklığı & Hacmi: Ortalama 18.68 günde bir alışveriş yaparlar, ortalama sipariş sayıları 8.95'tir.Zamanlama: Diğer tüm gruplardan belirgin şekilde ayrılarak, siparişlerini haftanın ilk günlerinde (Pazar / Pazartesi - avg_order_dow: 1.75) yoğunlaştırırlar.Pazarlama Aksiyonu: Pazar günleri akşamüstü veya Pazartesi sabahı "Haftalık Ev İhtiyaçları" ve "Haftaya Hazırlık" temalı alışveriş listesi hatırlatmaları ile satış hacmi yükseltilebilir.📊 Genel Özet TablosuKüme k (Cluster)Segment AdıMüşteri SayısıPazar Payı (%)Sipariş Aralığı (Gün)Toplam SiparişTercih Edilen Saat0Sadık & Yüksek Hacimli57.674%27.978.2535.8113:201Akşamüstü Alışverişçileri48.916%23.7217.819.2015:102Erken Saat Alışverişçileri50.896%24.6818.249.2311:303Hafta Başı Alışverişçileri48.723%23.63

In [ ]:
from google.cloud import bigquery

# 1. Segment isimlerini DataFrame'e ekleyelim
segment_mapping = {
    0: 'Sadik ve Yuksek Hacimli',
    1: 'Aksamustu Alisveriscileri',
    2: 'Sabah Alisveriscileri',
    3: 'Hafta Basi Alisveriscileri'
}

df_rfm['Segment_Name'] = df_rfm['Cluster'].map(segment_mapping)

# 2. Integer sütun tiplerini netleştirelim
df_rfm['Cluster'] = df_rfm['Cluster'].astype(int)
df_rfm['total_orders'] = df_rfm['total_orders'].astype(int)

# 3. BigQuery'ye Yükleme
table_id = f"{PROJECT_ID}.Instacart_Raw_Dataset.customer_segmentation_results"

job = client.load_table_from_dataframe(
    df_rfm,
    table_id,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
)

job.result()  # İşlemin tamamlanmasını bekle
print("Yükleme Başarılı!")

yetki hatası veriyor data set tablo olarak alıyorum

In [ ]:
# Segment isimlerini ekleyelim
segment_mapping = {
    0: 'Sadik ve Yuksek Hacimli',
    1: 'Aksamustu Alisveriscileri',
    2: 'Sabah Alisveriscileri',
    3: 'Hafta Basi Alisveriscileri'
}

df_rfm['Segment_Name'] = df_rfm['Cluster'].map(segment_mapping)

# Bilgisayarınıza CSV olarak indirin
df_rfm.to_csv('musteri_segmentleri_sonuc.csv', index=False)

from google.colab import files
files.download('musteri_segmentleri_sonuc.csv')

print("Müşteri segmentleri bilgisayarınıza indiriliyor!")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Segment Dağılımı Pasta Grafiği
fig1 = px.pie(
    df_rfm,
    names='Segment_Name',
    title='<b>Müşteri Segment Dağılımı (% Oranlar)</b>',
    color_discrete_sequence=px.colors.qualitative.Set3,
    hole=0.4
)
fig1.show()

# 2. Segmentlerin Sipariş Hacmi vs Sipariş Aralığı (Scatter Plot)
fig2 = px.scatter(
    df_rfm.sample(2000), # Hafif olması için 2000 örneklem
    x='avg_days_between_orders',
    y='total_orders',
    color='Segment_Name',
    title='<b>Sipariş Aralığı ve Sipariş Sayısı Dağılımı</b>',
    labels={'avg_days_between_orders': 'Ort. Sipariş Aralığı (Gün)', 'total_orders': 'Toplam Sipariş Sayısı'},
    hover_data=['user_id']
)
fig2.show()

gorseleştirme yapılısı